# 03 — Feature Engineering

Turn the raw, cleaned data into a modeling-ready dataset: add the engineered
features from `src/features.py`, split into train/val/test, then fit a
scaling pipeline on train only and save everything needed for
`04_modeling.ipynb`.

Structure:
1. Load and clean raw data
2. Carry forward findings from `02_statistical_analysis.ipynb`
3. Build engineered features
4. Train / validation / test split
5. Fit preprocessing pipeline (train only — no leakage)
6. Save processed datasets and pipeline artifact

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import joblib

from src.data_loader import load_raw_data
from src.utils import load_config, set_seed, resolve_path
from src.preprocessing import clean_data, split_data, build_preprocessing_pipeline
from src.features import engineer_features

config = load_config()
set_seed(config["random_seed"])
target = config["data"]["target_column"]

## 1. Load and Clean Raw Data

Reusing the same loading/cleaning steps as notebooks 01-02 — nothing new here,
just getting back to the validated starting point before adding features.

In [2]:
df_raw = load_raw_data()
df_clean = clean_data(df_raw)
print(df_clean.shape)
df_clean.head()

2026-09-17 15:49:26 | src.data_loader | INFO | Loading cached raw data from C:\diabetes-risk-prediction\data\raw\diabetes_health_indicators.csv
2026-09-17 15:49:26 | src.data_loader | INFO | Raw data validation passed.
2026-09-17 15:49:26 | src.preprocessing | INFO | Dropped 24206 duplicate rows (9.54%)


(229474, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40.0,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25.0,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28.0,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27.0,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24.0,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 2. Carrying Forward Notebook 02's Findings

`02_statistical_analysis.ipynb` ranked every feature by effect size and flagged
which survive multiple-testing correction. Loading that table here to make the
feature engineering decisions explicit and traceable, rather than re-deciding
from scratch.

**Decision:** keep all original features regardless of individual effect size.
Tree-based models (Random Forest, XGBoost) are not harmed by weak features —
they simply get low importance — and dropping them would only matter for
logistic regression, where regularization already handles this. What the
ranking *does* inform: which features get emphasized in the SHAP
interpretability notebook (05) and the risk segmentation notebook (06).

In [3]:
stats_results_path = Path("../reports/statistical_test_results.csv")
if stats_results_path.exists():
    stats_results = pd.read_csv(stats_results_path)
    print("Top 5 features by effect size (from notebook 02):")
    display(stats_results.head())
else:
    print("Run 02_statistical_analysis.ipynb first to generate reports/statistical_test_results.csv")

Top 5 features by effect size (from notebook 02):


,feature,test,p_value,effect_size,effect_size_measure,effect_size_label,p_value_corrected,significant_after_correction
0,BMI,mann-whitney-u,0.0,0.376633,rank_biserial,medium,0.0,True
1,GenHlth,chi-square,0.0,0.299296,cramers_v,medium,0.0,True
2,HighBP,chi-square,0.0,0.263110,cramers_v,medium,0.0,True
3,PhysHlth,mann-whitney-u,0.0,0.226029,rank_biserial,small,0.0,True
4,DiffWalk,chi-square,0.0,0.218321,cramers_v,medium,0.0,True


## 3. Build Engineered Features

Applying the three engineered features from `src/features.py`:
- `BMI_category` (one-hot encoded) — buckets BMI into clinical ranges, since
  risk doesn't necessarily rise linearly with raw BMI
- `UnwellDays` — combines MentHlth + PhysHlth into one composite wellness score
- `RiskFactorCount` — counts how many known binary risk factors a respondent has

This is done on the full cleaned dataset before splitting, since these are
deterministic transformations of each row's own values — they don't use any
information about the target or other rows, so there's no leakage risk in
doing this pre-split (unlike scaling or SMOTE, which must be fit post-split).

In [4]:
df_features = engineer_features(df_clean, encode=True)
new_cols = [c for c in df_features.columns if c not in df_clean.columns]
print("New columns added:", new_cols)
df_features[new_cols].head()

New columns added: ['UnwellDays', 'RiskFactorCount', 'BMI_category_underweight', 'BMI_category_normal', 'BMI_category_overweight', 'BMI_category_obese']


,UnwellDays,RiskFactorCount,BMI_category_underweight,BMI_category_normal,BMI_category_overweight,BMI_category_obese
0,30,3,0,0,0,1
1,0,1,0,0,1,0
2,30,2,0,0,1,0
3,0,1,0,0,1,0
4,3,2,0,1,0,0


## 4. Train / Validation / Test Split

Splitting now, before any fitting happens (scaling comes next) — this is the
line past which train, validation, and test data must never mix.

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df_features,
    target_column=target,
    test_size=config["data"]["test_size"],
    val_size=config["data"]["val_size"],
    random_state=config["random_seed"],
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

2026-09-17 15:49:27 | src.preprocessing | INFO | Split sizes — train: 160631, val: 22948, test: 45895


Train: (160631, 27), Val: (22948, 27), Test: (45895, 27)


## 5. Fit Preprocessing Pipeline (Train Only)

Scaling the continuous/count features (BMI, MentHlth, PhysHlth, UnwellDays,
RiskFactorCount) — needed for logistic regression, harmless for tree models.
Everything else (binary features, one-hot BMI_category columns, ordinal
scales) passes through unchanged.

Critical rule: `fit_transform` on train, `transform` only on val/test. Fitting
the scaler on anything but train would leak val/test statistics into the
features the model learns from.

In [6]:
numeric_features = ["BMI", "MentHlth", "PhysHlth", "UnwellDays", "RiskFactorCount"]
passthrough_features = [c for c in X_train.columns if c not in numeric_features]

preprocessor = build_preprocessing_pipeline(numeric_features, passthrough_features)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names = numeric_features + passthrough_features
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_val_df = pd.DataFrame(X_val_processed, columns=feature_names, index=X_val.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_df.shape, X_val_df.shape, X_test_df.shape)
X_train_df.head()

Processed shapes: (160631, 27) (22948, 27) (45895, 27)


,BMI,MentHlth,PhysHlth,UnwellDays,RiskFactorCount,HighBP,HighChol,CholCheck,Smoker,Stroke,...,GenHlth,DiffWalk,Sex,Age,Education,Income,BMI_category_underweight,BMI_category_normal,BMI_category_overweight,BMI_category_obese
216076,-0.841123,0.837606,-0.518087,0.279940,-0.485581,0.0,0.0,1.0,1.0,0.0,...,2.0,0.0,0.0,5.0,5.0,1.0,0.0,1.0,0.0,0.0
45712,0.934923,-0.455162,-0.518087,-0.669695,0.365123,1.0,1.0,1.0,0.0,0.0,...,4.0,0.0,0.0,10.0,6.0,7.0,0.0,0.0,0.0,1.0
79855,0.194904,-0.455162,2.796409,2.179211,0.365123,1.0,0.0,1.0,0.0,1.0,...,4.0,0.0,1.0,12.0,4.0,4.0,0.0,0.0,0.0,1.0
11567,-0.989127,-0.455162,-0.518087,-0.669695,-0.485581,0.0,0.0,1.0,1.0,0.0,...,2.0,0.0,1.0,7.0,4.0,3.0,0.0,1.0,0.0,0.0
78065,0.490911,3.423144,-0.076154,2.179211,-1.336286,0.0,0.0,1.0,0.0,0.0,...,4.0,0.0,0.0,5.0,5.0,1.0,0.0,0.0,0.0,1.0


In [7]:
# Sanity check for leakage: train's scaled numeric columns should be ~exactly
# standardized (mean 0, std 1); val/test should NOT be exactly 0/1, since
# they're transformed using train's fitted scaler, not their own.
print("Train BMI (scaled) mean/std:", X_train_df["BMI"].mean().round(4), X_train_df["BMI"].std().round(4))
print("Val BMI (scaled) mean/std:  ", X_val_df["BMI"].mean().round(4), X_val_df["BMI"].std().round(4))
print("Test BMI (scaled) mean/std: ", X_test_df["BMI"].mean().round(4), X_test_df["BMI"].std().round(4))
assert abs(X_train_df["BMI"].mean()) < 1e-6, "Train should be exactly standardized"
print("\nNo leakage detected — val/test are not exactly standardized, as expected.")

Train BMI (scaled) mean/std: 0.0 1.0
Val BMI (scaled) mean/std:   -0.0081 1.0132
Test BMI (scaled) mean/std:  0.0073 1.0174

No leakage detected — val/test are not exactly standardized, as expected.


## 6. Save Processed Datasets and Pipeline Artifact

Saving train/val/test splits (features + targets) as parquet, and the fitted
preprocessor as a joblib artifact — `04_modeling.ipynb` loads these directly
rather than re-running this notebook's logic, and the saved preprocessor is
exactly what a future deployment (see README's Future Improvements) would
reuse to transform new incoming data consistently.

In [8]:
processed_dir = resolve_path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train_df.to_parquet(processed_dir / "X_train.parquet")
X_val_df.to_parquet(processed_dir / "X_val.parquet")
X_test_df.to_parquet(processed_dir / "X_test.parquet")
y_train.to_frame().to_parquet(processed_dir / "y_train.parquet")
y_val.to_frame().to_parquet(processed_dir / "y_val.parquet")
y_test.to_frame().to_parquet(processed_dir / "y_test.parquet")

models_dir = resolve_path(config["paths"]["models_dir"])
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, models_dir / "preprocessor.joblib")

print("Saved processed train/val/test splits to data/processed/")
print("Saved fitted preprocessor to models/preprocessor.joblib")

Saved processed train/val/test splits to data/processed/
Saved fitted preprocessor to models/preprocessor.joblib


Next: `04_modeling.ipynb` loads these processed splits directly and trains
Logistic Regression, Random Forest, and XGBoost on them.